In [ ]:
import networkx as nx
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
import matplotlib.font_manager
from scipy.stats import pearsonr
from scipy.stats import linregress
from matplotlib import pyplot as plt
import matplotlib as mpl
from pycirclize import Circos

In [ ]:
matplotlib.font_manager.fontManager.addfont('/home/yzx46/PC_file/OncoNiche/Arial.ttf')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8 
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['axes.titlecolor'] = 'black'
plt.rcParams['legend.labelcolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.5

# Figure1A

In [ ]:
primary_mut_circos=pd.read_csv('primary_mut_circos_cancertype_color_v3.txt',sep='\t')
tissue_color=primary_mut_circos[['tissue','tissue_name'	,'tissue_cor_v2']].drop_duplicates()
tissue_name_rank=pd.DataFrame(primary_mut_circos['tissue_name'].value_counts())
tissue_name_rank['tissue_name']=tissue_name_rank.index.tolist()
tissue_name_rank.reset_index(drop=True,inplace=True)
tissue_sectors_df = primary_mut_circos[['tissue_name','tissue_cor_v2']].drop_duplicates()
tissue_sectors=tissue_sectors_df.set_index('tissue_name')['tissue_cor_v2'].to_dict()
tissue_count=pd.DataFrame(primary_mut_circos.value_counts('tissue_name'))
tissue_count.columns=['donor_size']
tissue_count['tissue_name']=tissue_count.index.tolist()
tissue_count.reset_index(drop=True,inplace=True)
tissue_count['Rank']=pd.Categorical(tissue_count['tissue_name'], categories=tissue_name_rank['tissue_name'].tolist(), ordered=True)
tissue_count = tissue_count.sort_values('Rank')
tissue_sectors=tissue_count.set_index('tissue_name')['donor_size'].to_dict()

project_color=primary_mut_circos[['oncotree_type_level_two','tissue_name','cancer_color_func_output','tissue_cor_v2']].drop_duplicates()
primary_mut_circos.columns.tolist()
tissue_eg='Thyroid'

In [ ]:
circos = Circos(tissue_sectors, space=0)
for sector in circos.sectors:
tissue_eg=sector.name
tissue_data_eg=primary_mut_circos[primary_mut_circos['tissue_name']==tissue_eg]
tissue_color_eg=''.join(tissue_data_eg['tissue_cor_v2'].drop_duplicates())
# Plot sector axis & name text
if (sector.name=='Ovary/Fallopian tube') | (sector.name=='Bladder/Urinary tract') |(sector.name=='Uterus') |(sector.name=='Myeloid') |(sector.name=='Lymphoid')|(sector.name=='Biliary tract')|(sector.name=='Adrenal gland')|(sector.name=='Skin')|(sector.name=='Cervix')|(sector.name=='Bone') :
    sector.text("", size=8)
else:
    sector.text(f"{sector.name}", size=8)
track1 = sector.add_track((85, 100))  
track1.axis(alpha=0.5,fc=tissue_color_eg) 
if (sector.size==251) | (sector.size==216):
    track1.text("", size=8)
else:
    track1.text(f"{sector.size}", size=8)
x_all = np.arange(0, int(sector.size), 1)
y_driver = tissue_data_eg['driver_mut_count'].tolist()
bar_track_driver = sector.add_track((70, 80))
bar_track_driver.axis()
bar_track_driver.bar(x_all, y_driver,width=0.8,color=tissue_color_eg)
y_tissue = [1]*len(x_all)
bar_track_tissue = sector.add_track((55, 65))
bar_track_tissue.axis()
bar_track_tissue.bar(x_all, y_tissue,width=1,color=tissue_data_eg['cancer_color_func_output'])

fig = plt.gcf()   # 获取当前 Figure
fig.savefig("circos.png", dpi=300, bbox_inches='tight', pad_inches=0.1)

In [ ]:
#######plot WGS and WES##########
project_tissue_count_data=pd.read_csv('project_plot_data.txt',sep='\t')
fig, axes = plt.subplots(1,1,figsize=(3.5,1.1))
plt.rcParams['axes.linewidth'] = 0.5
sns.barplot(data=project_tissue_count_data,x='count',y='projectnology',hue='tissue',palette=project_tissue_count_data['color'],ax=axes)
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.set_ylabel('Sample sizes',fontsize=8, fontname='Arial')
axes.set_yticklabels(project_tissue_count_data['projectnology'].drop_duplicates(),fontsize=8, fontname='Arial')
axes.set_xlabel('',fontsize=8, fontname='Arial')
axes.set_ylabel('',fontsize=8, fontname='Arial')
axes.legend_.remove()
plt.savefig('project_sample_size.pdf',dpi=300,bbox_inches='tight')

tech_tissue_count_data=pd.read_csv('technology_plot_data.txt',sep='\t')
fig, axes = plt.subplots(1,1,figsize=(3.5,1.1))
plt.rcParams['axes.linewidth'] = 0.5
sns.barplot(data=tech_tissue_count_data,x='count',y='technology',hue='tissue',palette=tech_tissue_count_data['color'],ax=axes)
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.set_yticklabels(tech_tissue_count_data['technology'].drop_duplicates(),fontsize=8, fontname='Arial')
axes.set_xlabel('',fontsize=8, fontname='Arial')
axes.set_ylabel('',fontsize=8, fontname='Arial')
axes.legend_.remove()
axes.tick_params(axis='x', width=0.5)
axes.tick_params(axis='y', width=0.5)
plt.savefig('tech_sample_size_v2.pdf',dpi=300,bbox_inches='tight')

In [ ]:
## expression donor number
primary_exp_donor_color_sort=pd.read_csv('/primary_exp_donor_color_data.txt',sep='\t')
fig, axes = plt.subplots(1,1,figsize=(8.5/4,8.5/4))
plt.rcParams['axes.linewidth'] = 0.5
sns.barplot(data=primary_exp_donor_color_sort,x='tissue_name',y='sample_sizes',palette=primary_exp_donor_color_sort['tissue_cor_v2'],ax=axes,width=1)
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.set_ylabel('Sample sizes',fontsize=8, fontname='Arial')
axes.set_xticklabels(primary_exp_donor_color_sort['tissue_name'],fontsize=8, fontname='Arial')
axes.set_xlabel('',fontsize=8, fontname='Arial')
axes.tick_params(labelrotation=90,axis='x')
plt.savefig('/primary_exp_donor_width_v2.pdf',dpi=300,bbox_inches='tight')
fig, ax = plt.subplots(figsize=(1.4,1.4))
# 准备数据
x = np.arange(len(primary_exp_donor_color_sort))
heights = primary_exp_donor_color_sort['sample_sizes']
colors = primary_exp_donor_color_sort['tissue_cor_v2']
# 使用matplotlib绘图，精确控制width
ax.bar(x, heights, color=colors, width=1)  # width越接近1，间距越小
ax.set_ylabel('Sample Size')
ax.set_xlabel('Tissue Name')
# 去除x轴和y轴刻度
ax.set_xticks([])
# 去除x轴和y轴标签（如果有）
ax.set_xlabel("")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Sample sizes',fontsize=8, fontname='Arial')
ax.set_xlabel('',fontsize=8, fontname='Arial')
ax.tick_params(labelrotation=90,axis='x')
plt.savefig('exp/primary_exp_donor_width_v2.pdf',dpi=300,bbox_inches='tight')

## Figure 1B

In [ ]:
driver_cancer_max_prevalance_plot_all_color=pd.read_csv('driver_cancer_max_prevalance_plot_all_color.txt',sep='\t')
driver_cancer_max_prevalance_plot_all_color_tissue_sort=driver_cancer_max_prevalance_plot_all_color.sort_values('tissue_num')[::-1].iloc[0:10,:]
driver_cancer_max_prevalance_plot_all_color_tissue_one=driver_cancer_max_prevalance_plot_all_color[driver_cancer_max_prevalance_plot_all_color['tissue_num']==1].sort_values('max_log10')
primary_mut_circos=pd.read_csv('primary_mut_circos_cancertype_color_v3.txt',sep='\t')
mutagene_input=pd.read_csv('mutagene_input_all_v2.txt',sep='\t')
mutagene_input_tissue=primary_mut_circos[primary_mut_circos['submitted_donor_id'].isin(mutagene_input['Tumor_Sample_Barcode'])]
mutagene_input_tissue_counts=pd.DataFrame(mutagene_input_tissue['tissue_name'].value_counts())
mutagene_input_tissue_counts['tissue_name']=mutagene_input_tissue_counts.index.tolist()
mutagene_input_tissue_counts.reset_index(drop=True,inplace=True)
driver_cancer_max_prevalance_plot_all_color_rate=pd.merge(driver_cancer_max_prevalance_plot_all_color,mutagene_input_tissue_counts,on='tissue_name')
driver_cancer_max_prevalance_plot_all_color_rate['frequence']=driver_cancer_max_prevalance_plot_all_color_rate['max_mut_count']/driver_cancer_max_prevalance_plot_all_color_rate['count']

tissue_color=pd.read_csv('tissue_color.txt',sep='\t')
color_palette=tissue_color[['tissue_name','tissue_cor_v2']].drop_duplicates()
color_palette=dict(zip(color_palette['tissue_name'],color_palette['tissue_cor_v2']))

fig, axes = plt.subplots(1,1,figsize=(8.5/2,3.55))
plt.rcParams['axes.linewidth'] = 0.5
sns.scatterplot(data=driver_cancer_max_prevalance_plot_all_color_rate,y='frequence',x='tissue_num',ax=axes,hue='tissue_name',palette=color_palette,legend=False,s=10)
axes.set_ylabel(r'Max mutation frequency',fontsize=8, fontname='Arial')
axes.set_xlabel('Number of driver cancer types',fontsize=8, fontname='Arial')
# 设置x轴刻度范围和刻度
axes.set_xlim(0, 22)
axes.set_xticks(range(0, 23, 5))
# 设置y轴起始位置为0
axes.set_ylim(0, axes.get_ylim()[1])
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.tick_params(axis='x', labelsize=8)
axes.tick_params(axis='y', labelsize=8)
plt.savefig('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/driver_mut_distribution_gini/driver_mut_distribution_mutation_frequency.pdf',dpi=300,bbox_inches='tight')


## Figure1C

In [ ]:
tissue_color=pd.read_csv('donor_color.txt',sep='\t')
tissue_color_pro=tissue_color[['sample_size_primary','tissue','tissue_name','tissue_cor_v2']]
driver_gini=pd.read_csv('gini_count4_driver_tissue_v2.txt',sep='\t')
driver_gini_color=pd.merge(driver_gini,tissue_color_pro,left_on='Tissue',right_on='tissue')
driver_gini_distri=driver_gini_color[['Driver_Hgvsp','Gini']].drop_duplicates()
driver_gini_distri.columns=['Driver mutations','Gini index']

fig,axes=plt.subplots(1,1,figsize=(9.25,1.3))
plt.subplots_adjust(wspace=0.4)
plt.rcParams['axes.linewidth'] = 0.5
sns.kdeplot(x=driver_gini_distri['Gini'],shade=True,color="#EE7B6C",ax=axes)
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.set_ylabel('Probability density',fontsize=8, fontname='Arial')
axes.set_xlabel('Gini index',fontsize=8, fontname='Arial')
axes.axvline(x=0.8, color='#80AACA', linestyle='--')
axes.set_xlim(0.6,1)
axes.tick_params(axis='x', labelsize=8)
axes.tick_params(axis='y', labelsize=8)
plt.savefig(f'mut_kdeplot_v2.pdf',dpi=300,bbox_inches='tight')


## Figure1D

In [ ]:
## top20 TS mutation
tissue_color=pd.read_csv('/tissue_color.txt',sep='\t')
driver_gene_color=pd.read_csv('ts_mut_tissue_count_top20.txt',sep='\t')
driver_gene_color.drop(['tissue_cor_v2'],axis=1,inplace=True)
driver_gene_color=pd.merge(driver_gene_color,tissue_color,on='tissue_name')
driver_gene_color.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/top20_TS_mut/ts_mut_tissue_count_top20_v2.txt',sep='\t',index=False)
driver_gene_color[driver_gene_color['gene']=='BRAF']
ts_mut_number=data.frame(read.table('ts_mut_tissue_count_top20_v2.txt',sep='\t',header=1,comment.char = "",, check.names = FALSE))

ts_mut_number_tissue_sort_all=NULL
for (gene_eg in unique(ts_mut_number$gene)){
  ts_mut_number_eg=ts_mut_number[which(ts_mut_number$gene==gene_eg),]
  ts_mut_number_eg_sort=ts_mut_number_eg[order(ts_mut_number_eg$count),]
  ts_mut_number_tissue_sort_all=rbind(ts_mut_number_eg_sort,ts_mut_number_tissue_sort_all)
}
ts_mut_number_sum=ts_mut_number %>%
  group_by(gene) %>%
  summarise(total_gene_number = sum(count))
ts_mut_number_sort=ts_mut_number_sum[order(ts_mut_number_sum$total_gene_number,decreasing = F),]

ts_mut_number$gene=factor(ts_mut_number$gene, levels = unique(ts_mut_number_sort$gene))

color_all=ts_mut_number$tissue_cor_v2
tissue_all=ts_mut_number$tissue_name
names(color_all)=tissue_all


pdf('ts_mut_gene_number_top20.pdf',width=8.5/2, height=11/3)
ggplot(data = ts_mut_number,aes(x=count,y=gene,group=tissue_name  ))+
  geom_bar(stat = "identity" , position="stack", width =0.5 ,aes(fill=tissue_name ) ) +  theme_minimal()+
  scale_fill_manual(values = color_all)+ ##用于手动设置离散型填充颜色的函数
  theme(legend.position = "none",    
        axis.line.y = element_line(size = (0.5/1.07)*0.5),
        axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
        axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
        panel.grid=element_blank(),
        axis.text.x = element_text(size = 8, family = "sans", color = "black"),
        axis.text.y = element_text(size = 8, family = "sans", color = "black") # 调整 x 轴刻度线宽度) + # 去除 y 轴刻度线
  )+ # 去除 y 轴刻度线
  xlab('The number of tissue-specific mutations')+ylab('The top 20 driver genes')+
  font("xlab",size = 8, family = "sans", color = "black")+
  font("ylab",size = 8, family = "sans", color = "black")+
  scale_y_discrete(expand = c(0, 0)) +  # 确保 y 轴从原点开始
  scale_x_continuous(expand = c(0, 0), limits = c(0, 160))
dev.off()

## Figure1E

In [ ]:
mut_exp_number=pd.read_csv('primary_ts_mut_exp_number.txt',sep='\t')
mut_exp_number.sort_values('ts_mut_gene')
mut_exp_number['ts_exp_gene'].mean()
tissue_type_name=pd.read_csv('tissue_type_name.txt',sep='\t')
tissue_type_name_process=tissue_type_name[['tissue','tissue_name']]
mut_exp_number=pd.merge(mut_exp_number,tissue_type_name_process,on='tissue')
mut_exp_number_sort=mut_exp_number.sort_values('ts_mut_gene')

mut_exp_num_df_all=pd.DataFrame()
for pos in range(0,mut_exp_number_sort.shape[0]):
    mut_exp_number_sort_eg=mut_exp_number_sort.iloc[pos:(pos+1),]
    tissue_eg=''.join(mut_exp_number_sort_eg['tissue_name_y'])
    ts_mut_gene_num_eg=int(mut_exp_number_sort_eg['ts_mut_gene'])
    ts_exp_gene_num_eg=int(mut_exp_number_sort_eg['ts_exp_gene'])
    mut_exp_num_df_eg=pd.DataFrame()
    mut_exp_num_df_eg['Tissue']=[tissue_eg]*2
    mut_exp_num_df_eg['Group']=['Tissue-specific genetic mutations','Tissue-specific expressed genes']
    mut_exp_num_df_eg['Number']=[ts_mut_gene_num_eg,ts_exp_gene_num_eg]
    mut_exp_num_df_all=pd.concat([mut_exp_num_df_eg,mut_exp_num_df_all])

custom_palette = {"Tissue-specific genetic mutations": "#80AACA", "Tissue-specific expressed genes": "#EE7B6C"}  # You can customize colors here
mut_num_df_all=mut_exp_num_df_all[mut_exp_num_df_all['Group']=='Tissue-specific genetic mutations']
mut_num_df_all_sort=mut_num_df_all.sort_values('Number')

fig,axes=plt.subplots(1,1,figsize=(8.5/2,(11/8)/3*2))
plt.rcParams['axes.linewidth'] = 1.5
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
sns.set_style("ticks")
# 绘制双柱状图
sns.barplot(x='Tissue', y='Number', data=mut_num_df_all_sort, color='#7A9FBB',ax=axes)
# 设置Y轴标签
axes.set_ylabel('Number',fontsize=8, fontname='Arial')
axes.set_xlabel('')
# 设置标题
axes.set_title('',fontsize=8, fontname='Arial')
axes.set_xticklabels(mut_exp_num_df_all['Tissue'].drop_duplicates().tolist(),fontsize=8, fontname='Arial')
axes.tick_params(labelrotation=90,axis='x')
# 显示图例
plt.savefig(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_mut_gene_number_primary.pdf',dpi=300,bbox_inches='tight')

exp_num_df_all=mut_exp_num_df_all[mut_exp_num_df_all['Group']=='Tissue-specific expressed genes']
exp_num_df_all_sort=exp_num_df_all.sort_values('Number')
fig,axes=plt.subplots(1,1,figsize=(8.5/2,(11/8)/3*2))
plt.rcParams['axes.linewidth'] = 1.5
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
sns.set_style("ticks")
# 绘制双柱状图
sns.barplot(x='Tissue', y='Number', data=exp_num_df_all_sort, color='#EE7B6C',ax=axes)
# 设置Y轴标签
axes.set_ylabel('Number',fontsize=8, fontname='Arial')
axes.set_xlabel('')
# 设置标题
axes.set_title('',fontsize=8, fontname='Arial')
axes.set_xticklabels(exp_num_df_all_sort['Tissue'].drop_duplicates().tolist(),fontsize=8, fontname='Arial')
axes.tick_params(labelrotation=90,axis='x')
# 显示图例
plt.savefig(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_exp_gene_number_primary.pdf',dpi=300,bbox_inches='tight')

mut_num_df_all_sort=mut_num_df_all.sort_values('Tissue')
exp_num_df_all_sort=exp_num_df_all.sort_values('Tissue')
fig,axes=plt.subplots(2,1,figsize=(8.5/2,11/4))
plt.rcParams['axes.linewidth'] = 0.5
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
sns.set_style("ticks")
# 绘制双柱状图
sns.barplot(x='Tissue', y='Number', data=mut_num_df_all_sort, color='#7A9FBB',ax=axes[0])
# 设置Y轴标签
axes[0].set_ylabel('Number',fontsize=8, fontname='Arial')
axes[0].set_xlabel('')
# 设置标题
axes[0].set_title('',fontsize=8, fontname='Arial')
axes[0].set_xticklabels(mut_num_df_all_sort['Tissue'].drop_duplicates().tolist(),fontsize=8, fontname='Arial')
axes[0].tick_params(bottom=False, labelbottom=False)
# 显示图例
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
sns.set_style("ticks")
# 绘制双柱状图
sns.barplot(x='Tissue', y='Number', data=exp_num_df_all_sort, color='#EE7B6C',ax=axes[1])
# 设置Y轴标签
axes[1].set_ylabel('Number',fontsize=8, fontname='Arial')
axes[1].set_xlabel('')
# 设置标题
axes[1].set_title('',fontsize=8, fontname='Arial')
axes[1].set_xticklabels(exp_num_df_all_sort['Tissue'].drop_duplicates().tolist(),fontsize=8, fontname='Arial')
axes[1].tick_params(labelrotation=90,axis='x')
plt.savefig(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/ts_mut_exp_gene_number/ts_mut_exp_gene_number_primary.pdf',dpi=300,bbox_inches='tight')
